In [1]:
from pathlib import Path
from typing import Any, Callable, Dict, Optional, Sequence, Tuple

import numpy as np

from olbootstrap.study._study import UniformCoverageStudy
from olbootstrap.synthetic_time_series._ar1 import AR1Process

/opt/anaconda3/envs/bootstrap/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def run_sweeps_for_process_list(
    processes: Sequence[object],
    *,
    labels: Optional[Sequence[str]] = None,
    outdir: Path = Path("..") / "experiments",
    seed: Optional[int] = None,
    sample_size: int = 2000,
    smoothing_grid=(2 / 20, 2 / 50, 2 / 100, 2 / 250, 2 / 500),
    n_series: int = 100,
    burn_in: int = 500,
    alpha: float = 0.05,
    base_exp_kwargs: Optional[dict] = None,
    exp_kwargs_overrides: Optional[list] = None,
    progress: bool = True,
    parallel: bool = True,
    n_jobs: int = -1,
    var_warmup: int = 0,
    transform: str = "student",
    run_sweeps_kwargs: Optional[Dict[str, Any]] = None,
) -> Dict[str, Dict[str, Any]]:
    """Run UniformCoverageStudy.run_sweeps for each pre-built process in `processes`.

    Returns:
        dict[label] -> result (whatever UniformCoverageStudy.run_sweeps returns for that process)
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    labels = (
        list(labels) if labels is not None else [type(p).__name__ for p in processes]
    )
    if len(labels) != len(processes):
        raise ValueError("labels length must match processes length")

    run_sweeps_kwargs = {} if run_sweeps_kwargs is None else dict(run_sweeps_kwargs)
    base_exp_kwargs = {} if base_exp_kwargs is None else dict(base_exp_kwargs)
    exp_kwargs_overrides = exp_kwargs_overrides or [{}]

    ss = np.random.SeedSequence(seed) if seed is not None else None
    child_seqs = ss.spawn(len(processes)) if ss is not None else [None] * len(processes)

    results: Dict[str, Dict[str, Any]] = {}
    for i, (proc, label) in enumerate(zip(processes, labels)):
        proc_outdir = outdir / label
        proc_outdir.mkdir(parents=True, exist_ok=True)
        this_seed = None if child_seqs[i] is None else int(child_seqs[i].entropy)

        print(
            f"[run_sweeps_for_process_list] Running sweeps for {label} -> saving to {proc_outdir}"
        )

        res = UniformCoverageStudy.run_sweeps(
            base_process_template=proc,
            sample_size=sample_size,
            dgp_overrides=None,  # you said you predefine the DGPs — no overrides needed
            exp_kwargs_overrides=exp_kwargs_overrides,
            smoothing_grid=smoothing_grid,
            outdir=proc_outdir,
            n_series=n_series,
            burn_in=burn_in,
            alpha=alpha,
            base_exp_kwargs=base_exp_kwargs,
            seed=this_seed,
            progress=progress,
            parallel=parallel,
            n_jobs=n_jobs,
            transform=transform,
            var_warmup=var_warmup,
            **run_sweeps_kwargs,  # any extra args you want to forward
        )

        results[label] = res

    return results

In [ ]:
def run_param_sweep(
    make_processes: Callable[[Any, np.random.Generator], Tuple[Sequence, Sequence]],
    param_values: Sequence,
    param_name: str,
    base_outdir: Path,
    run_sweeps_common_kwargs: Dict[str, Any],
    run_kwargs_fn: Optional[Callable[[Any], Dict[str, Any]]] = None,
    seed_master: int = 1234,
    seed_per_value: bool = True,
    per_value_subfolder: bool = True,
    progress: bool = True,
    parallel: bool = True,
    n_jobs: int = -1,
) -> Dict[Any, Any]:
    base_outdir = Path(base_outdir)
    base_outdir.mkdir(parents=True, exist_ok=True)

    results_map = {}
    for idx, val in enumerate(param_values):
        seed_for_run = seed_master + idx if seed_per_value else seed_master
        rng = np.random.default_rng(seed_for_run)

        processes, labels = make_processes(val, rng)

        if per_value_subfolder:
            run_outdir = base_outdir / f"{param_name}_{val}"
            run_outdir.mkdir(parents=True, exist_ok=True)
        else:
            run_outdir = base_outdir

        run_kwargs = dict(run_sweeps_common_kwargs)  # shallow copy
        run_kwargs.setdefault("progress", progress)
        run_kwargs.setdefault("parallel", parallel)
        run_kwargs.setdefault("n_jobs", n_jobs)

        if run_kwargs_fn is not None:
            extra = run_kwargs_fn(val) or {}
            run_kwargs.update(extra)

        res_map = run_sweeps_for_process_list(
            processes,
            labels=labels,
            outdir=run_outdir,
            seed=seed_for_run,
            **run_kwargs,
        )

        results_map[val] = res_map
        print(f"[done] {param_name}={val} -> saved in {run_outdir}")

    return results_map

In [ ]:
phi = 0.5
tp_values = [1 / 4, 1 / 3, 1 / 2, 1.0, 2.0, 3.0]  # transform powers
seed_master = 1234

sample_size = 3500
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 150
burn_in = 500
alpha = 0.1
var_warmup_fixed = 400
B_fixed = 200
transform = "student"

base_outdir = Path("..") / "experiments" / "AR1_phi0p5_tp_sweep"

common_kwargs = {
    "sample_size": sample_size,
    "smoothing_grid": smoothing_grid,
    "n_series": n_series,
    "burn_in": burn_in,
    "alpha": alpha,
    "var_warmup": var_warmup_fixed,
    "transform": transform,
    "base_exp_kwargs": {
        "B": B_fixed,
        "record_every": 1,
        "smoothing_method": "ewma",
    },
}


def make_ar1_phi_fixed(_tp, rng):
    proc = AR1Process(
        mean=0.0,
        phi=phi,
        noise_std=1.0,
        trend_slope=0.0,
        seasonal_amplitude=0.0,
        seasonal_period=None,
        seasonal_phase=0.0,
        rng=rng,  # different RNG per sweep value (via run_param_sweep)
    )
    label = rf"AR(1) $\phi={phi:g}$ (tp={_tp:g})"
    return [proc], [label]


def rkf(tp):
    be = dict(common_kwargs["base_exp_kwargs"])
    be["transform_power"] = float(tp)
    return {"base_exp_kwargs": be}


res_tp_sweep = run_param_sweep(
    make_processes=make_ar1_phi_fixed,
    param_values=tp_values,
    param_name="tp",
    base_outdir=base_outdir,
    run_sweeps_common_kwargs=common_kwargs,
    run_kwargs_fn=rkf,  # per-value: set transform_power
    seed_master=seed_master,
    seed_per_value=True,
    per_value_subfolder=True,  # folders like tp_0p25, tp_0p333333..., etc.
    progress=True,
    parallel=True,
    n_jobs=-1,
)

[run_sweeps_for_process_list] Running sweeps for AR(1) $\phi=0.5$ (tp=0.25) -> saving to ../experiments/AR1_phi0p5_tp_sweep/tp_0.25/AR(1) $\phi=0.5$ (tp=0.25)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  3.1min remaining:  4.7min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  3.1min remaining:  2.1min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.2min finished
combo runs: 100%|██████████| 1/1 [03:10<00:00, 190.51s/it]


[done] tp=0.25 -> saved in ../experiments/AR1_phi0p5_tp_sweep/tp_0.25
[run_sweeps_for_process_list] Running sweeps for AR(1) $\phi=0.5$ (tp=0.333333) -> saving to ../experiments/AR1_phi0p5_tp_sweep/tp_0.3333333333333333/AR(1) $\phi=0.5$ (tp=0.333333)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  3.1min remaining:  4.6min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  3.1min remaining:  2.1min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.2min finished
combo runs: 100%|██████████| 1/1 [03:10<00:00, 190.24s/it]


...

[done] tp=1.0 -> saved in ../experiments/AR1_phi0p5_tp_sweep/tp_1.0
[run_sweeps_for_process_list] Running sweeps for AR(1) $\phi=0.5$ (tp=2) -> saving to ../experiments/AR1_phi0p5_tp_sweep/tp_2.0/AR(1) $\phi=0.5$ (tp=2)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  2.0min remaining:  3.0min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  2.0min remaining:  1.4min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.3min finished
combo runs: 100%|██████████| 1/1 [02:18<00:00, 138.80s/it]


[done] tp=2.0 -> saved in ../experiments/AR1_phi0p5_tp_sweep/tp_2.0
[run_sweeps_for_process_list] Running sweeps for AR(1) $\phi=0.5$ (tp=3) -> saving to ../experiments/AR1_phi0p5_tp_sweep/tp_3.0/AR(1) $\phi=0.5$ (tp=3)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  2.0min remaining:  3.1min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  2.1min remaining:  1.4min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.1min finished
combo runs: 100%|██████████| 1/1 [02:08<00:00, 128.17s/it]

[done] tp=3.0 -> saved in ../experiments/AR1_phi0p5_tp_sweep/tp_3.0


In [ ]:
phis = [0.2, 0.5, 0.7]

outdir = Path("..") / "experiments"
outdir.mkdir(parents=True, exist_ok=True)

seed_master = 1234
sample_size = 3500
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 120
burn_in = 500
alpha = 0.1
var_warmup = 400
transform = "student"
base_exp_kwargs = {"B": 200, "record_every": 1, "smoothing_method": "ewma"}
progress = True
parallel = True
n_jobs = -1

for idx_phi, phi in enumerate(phis):
    rng = np.random.default_rng(seed_master + idx_phi)

    ar1_base = AR1Process(
        mean=0.0,
        phi=phi,
        noise_std=1,
        trend_slope=0.0,
        seasonal_amplitude=0.0,
        seasonal_period=None,
        seasonal_phase=0.0,
        rng=rng,
    )

    ar1_trend = AR1Process(
        mean=0.0,
        phi=phi,
        noise_std=1,
        trend_slope=0.001,
        seasonal_amplitude=0.0,
        seasonal_period=None,
        seasonal_phase=0.0,
        rng=rng,
    )

    ar1_shock = AR1Process(
        mean=0.0,
        phi=phi,
        noise_std=1,
        trend_slope=0.001,
        seasonal_amplitude=0.0,
        seasonal_period=None,
        seasonal_phase=0.0,
        rng=rng,
        shock_type="permanent",
        jump_prob=0.005,
        jump_scale=2,
    )

    ar1_season = AR1Process(
        mean=0.0,
        phi=phi,
        noise_std=1,
        trend_slope=0.001,
        seasonal_amplitude=0.4,
        seasonal_period=400,
        seasonal_phase=0.0,
        rng=rng,
    )

    processes = [ar1_base, ar1_trend, ar1_shock, ar1_season]
    labels = [
        f"AR(1) $\\phi={phi:g}$ (base)",
        f"AR(1) $\\phi={phi:g}$ (trend)",
        f"AR(1) $\\phi={phi:g}$ (shocks)",
        f"AR(1) $\\phi={phi:g}$ (season)",
    ]

    processes = [ar1_base]
    labels = [
        f"AR(1) $\\phi={phi:g}$ (base)",
    ]

    res_map = run_sweeps_for_process_list(
        processes,
        labels=labels,
        outdir=outdir,
        seed=seed_master + idx_phi,
        sample_size=sample_size,
        smoothing_grid=smoothing_grid,
        n_series=n_series,
        burn_in=burn_in,
        alpha=alpha,
        var_warmup=var_warmup,
        transform=transform,
        base_exp_kwargs=base_exp_kwargs,
        progress=progress,
        parallel=parallel,
        n_jobs=n_jobs,
    )

    print(f"Finished phi={phi:g}")

[run_sweeps_for_process_list] Running sweeps for AR(1) $\phi=0.5$ (base) -> saving to ../experiments/AR(1) $\phi=0.5$ (base)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  2.6min remaining:  3.8min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  2.6min remaining:  1.7min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.7min finished
combo runs: 100%|██████████| 1/1 [02:39<00:00, 159.12s/it]


Finished phi=0.5
[run_sweeps_for_process_list] Running sweeps for AR(1) $\phi=0.7$ (base) -> saving to ../experiments/AR(1) $\phi=0.7$ (base)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  2.5min remaining:  3.8min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  2.5min remaining:  1.7min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.6min finished
combo runs: 100%|██████████| 1/1 [02:35<00:00, 155.39s/it]

Finished phi=0.7


In [15]:
phi = 0.5

# B values to sweep
B_values = [20, 50, 100, 150, 200, 300]

outdir = Path("..") / "experiments" / "AR(1) B (base)"
outdir.mkdir(parents=True, exist_ok=True)

seed_master = 1234
sample_size = 3500
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 150
burn_in = 500
alpha = 0.1
var_warmup = 400
transform = "student"
base_exp_kwargs = {"B": 200, "record_every": 1, "smoothing_method": "ewma"}
progress = True
parallel = True
n_jobs = -1

for idx, B in enumerate(B_values):
    rng = np.random.default_rng(seed_master + idx)

    ar1_base = AR1Process(
        mean=0.0,
        phi=phi,
        noise_std=1,
        trend_slope=0.0,
        seasonal_amplitude=0.0,
        seasonal_period=None,
        seasonal_phase=0.0,
        rng=rng,
    )

    processes = [ar1_base]
    labels = [f"AR(1) (base)"]

    # copy base args and override B for this run
    run_base_exp_kwargs = dict(base_exp_kwargs)
    run_base_exp_kwargs["B"] = B

    print(f"Running phi={phi:g}, B={B} -> saving into {outdir}")
    res_map = run_sweeps_for_process_list(
        processes,
        labels=labels,
        outdir=outdir,
        seed=seed_master
        + idx,  # keep this if you want deterministic but different seeds
        sample_size=sample_size,
        smoothing_grid=smoothing_grid,
        n_series=n_series,
        burn_in=burn_in,
        alpha=alpha,
        var_warmup=var_warmup,
        transform=transform,
        base_exp_kwargs=run_base_exp_kwargs,
        progress=progress,
        parallel=parallel,
        n_jobs=n_jobs,
    )

    print(f"Finished B={B}")

Running phi=0.5, B=20 -> saving into ../experiments/AR(1) B (base)
[run_sweeps_for_process_list] Running sweeps for AR(1) (base) -> saving to ../experiments/AR(1) B (base)/AR(1) (base)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  1.2min remaining:  1.9min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  1.2min remaining:   49.6s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  1.3min finished
combo runs: 100%|██████████| 1/1 [01:15<00:00, 75.07s/it]


Finished B=20
Running phi=0.5, B=50 -> saving into ../experiments/AR(1) B (base)
[run_sweeps_for_process_list] Running sweeps for AR(1) (base) -> saving to ../experiments/AR(1) B (base)/AR(1) (base)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  1.5min remaining:  2.3min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  1.5min remaining:  1.0min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  1.5min finished
combo runs: 100%|██████████| 1/1 [01:32<00:00, 92.67s/it]


Finished B=50
Running phi=0.5, B=100 -> saving into ../experiments/AR(1) B (base)
[run_sweeps_for_process_list] Running sweeps for AR(1) (base) -> saving to ../experiments/AR(1) B (base)/AR(1) (base)


...

Finished B=150
Running phi=0.5, B=200 -> saving into ../experiments/AR(1) B (base)
[run_sweeps_for_process_list] Running sweeps for AR(1) (base) -> saving to ../experiments/AR(1) B (base)/AR(1) (base)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  3.1min remaining:  4.7min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  3.2min remaining:  2.1min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.2min finished
combo runs: 100%|██████████| 1/1 [03:13<00:00, 193.23s/it]


Finished B=200
Running phi=0.5, B=300 -> saving into ../experiments/AR(1) B (base)
[run_sweeps_for_process_list] Running sweeps for AR(1) (base) -> saving to ../experiments/AR(1) B (base)/AR(1) (base)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  4.2min remaining:  6.4min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  4.3min remaining:  2.9min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  4.4min finished
combo runs: 100%|██████████| 1/1 [04:23<00:00, 263.53s/it]

Finished B=300


In [17]:
phi = 0.5

# var_warmup values to sweep
var_warmup_values = [20, 50, 100, 200, 400, 800]

# base outdir (each run will be saved into a subfolder for clarity)
base_outdir = Path("..") / "experiments" / "AR(1)_cal_(base)"
base_outdir.mkdir(parents=True, exist_ok=True)

seed_master = 1234
sample_size = 3500
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 150
burn_in = 500
alpha = 0.1
# var_warmup will be overridden per-loop
transform = "student"
base_exp_kwargs = {"B": 200, "record_every": 1, "smoothing_method": "ewma"}
progress = True
parallel = True
n_jobs = -1

for idx, var_warmup in enumerate(var_warmup_values):
    rng = np.random.default_rng(seed_master + idx)

    # create per-run subfolder so files don't collide
    outdir = base_outdir / f"vw_{var_warmup}"
    outdir.mkdir(parents=True, exist_ok=True)

    ar1_base = AR1Process(
        mean=0.0,
        phi=phi,
        noise_std=1,
        trend_slope=0.0,
        seasonal_amplitude=0.0,
        seasonal_period=None,
        seasonal_phase=0.0,
        rng=rng,
    )

    processes = [ar1_base]
    labels = [f"AR(1) (cal)"]

    print(f"Running phi={phi:g}, var_warmup={var_warmup} -> saving into {outdir}")
    res_map = run_sweeps_for_process_list(
        processes,
        labels=labels,
        outdir=outdir,
        seed=seed_master + idx,
        sample_size=sample_size,
        smoothing_grid=smoothing_grid,
        n_series=n_series,
        burn_in=burn_in,
        alpha=alpha,
        var_warmup=var_warmup,  # <-- vary this per-run
        transform=transform,
        base_exp_kwargs=base_exp_kwargs,
        progress=progress,
        parallel=parallel,
        n_jobs=n_jobs,
    )

    print(f"Finished var_warmup={var_warmup}")

Running phi=0.5, var_warmup=20 -> saving into ../experiments/AR(1)_cal_(base)/vw_20
[run_sweeps_for_process_list] Running sweeps for AR(1) (cal) -> saving to ../experiments/AR(1)_cal_(base)/vw_20/AR(1) (cal)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  3.1min remaining:  4.6min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  3.1min remaining:  2.1min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.2min finished
combo runs: 100%|██████████| 1/1 [03:10<00:00, 190.91s/it]


Finished var_warmup=20
Running phi=0.5, var_warmup=50 -> saving into ../experiments/AR(1)_cal_(base)/vw_50
[run_sweeps_for_process_list] Running sweeps for AR(1) (cal) -> saving to ../experiments/AR(1)_cal_(base)/vw_50/AR(1) (cal)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  3.1min remaining:  4.6min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  3.1min remaining:  2.1min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.2min finished
combo runs: 100%|██████████| 1/1 [03:10<00:00, 190.57s/it]


Finished var_warmup=50
Running phi=0.5, var_warmup=100 -> saving into ../experiments/AR(1)_cal_(base)/vw_100
[run_sweeps_for_process_list] Running sweeps for AR(1) (cal) -> saving to ../experiments/AR(1)_cal_(base)/vw_100/AR(1) (cal)


...

Finished var_warmup=200
Running phi=0.5, var_warmup=400 -> saving into ../experiments/AR(1)_cal_(base)/vw_400
[run_sweeps_for_process_list] Running sweeps for AR(1) (cal) -> saving to ../experiments/AR(1)_cal_(base)/vw_400/AR(1) (cal)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  3.1min remaining:  4.7min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  3.2min remaining:  2.1min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.2min finished
combo runs: 100%|██████████| 1/1 [03:13<00:00, 193.95s/it]


Finished var_warmup=400
Running phi=0.5, var_warmup=800 -> saving into ../experiments/AR(1)_cal_(base)/vw_800
[run_sweeps_for_process_list] Running sweeps for AR(1) (cal) -> saving to ../experiments/AR(1)_cal_(base)/vw_800/AR(1) (cal)


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  3.2min remaining:  4.7min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  3.2min remaining:  2.1min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  3.3min finished
combo runs: 100%|██████████| 1/1 [03:16<00:00, 196.84s/it]

Finished var_warmup=800


In [6]:
from olbootstrap.synthetic_time_series._garch import GARCH11Process

seed_base = 42
rng_base = np.random.default_rng(seed_base)

mean = 0.0
persistence = 0.98  # α + β
alpha_garch = 0.08  # how reactive to shocks (try 0.05–0.15)
beta_garch = persistence - alpha_garch
target_var = 1.0  # long-run variance you want
omega_garch = (1 - persistence) * target_var  # ensures E[h_t]=target_var

# Variant 1: stationary GARCH(1,1)
garch_base = GARCH11Process(
    mean=mean,
    omega=omega_garch,
    alpha=alpha_garch,
    beta=beta_garch,
    trend_slope=0.0,
    seasonal_amplitude=0.0,
    seasonal_period=None,
    seasonal_phase=0.0,
    noise_std=1,
    rng=np.random.default_rng(seed_base),
)

garch_trend = GARCH11Process(
    mean=mean,
    omega=omega_garch,
    alpha=alpha_garch,
    beta=beta_garch,
    trend_slope=0.001,  # small linear trend — tune as desired
    seasonal_amplitude=0.0,
    seasonal_period=None,
    seasonal_phase=0.0,
    noise_std=1,
    rng=np.random.default_rng(seed_base),
)

garch_shock = GARCH11Process(
    mean=mean,
    omega=omega_garch,
    alpha=alpha_garch,
    beta=beta_garch,
    trend_slope=0.001,
    seasonal_amplitude=0.0,
    seasonal_period=None,
    seasonal_phase=0.0,
    noise_std=1,
    rng=np.random.default_rng(seed_base),
    shock_type="permanent",  # permanent means the level L_i accumulates jumps
    jump_prob=0.005,  # probability of a jump at each time step
    jump_scale=2,  # Std dev of jump sizes J_i ~ N(0, jump_scale^2)
)

garch_season = GARCH11Process(
    mean=mean,
    omega=omega_garch,
    alpha=alpha_garch,
    beta=beta_garch,
    trend_slope=0.001,
    seasonal_amplitude=0.4,  # amplitude of sinusoidal seasonality
    seasonal_period=400,  # long period (tune to your experiment)
    seasonal_phase=0.0,
    noise_std=1,
    rng=np.random.default_rng(seed_base),
)

garch_processes = [garch_base, garch_trend, garch_shock, garch_season]
garch_labels = ["GARCH_base", "GARCH_trend", "GARCH_shock", "GARCH_season"]

In [7]:
res_map = run_sweeps_for_process_list(
    garch_processes,
    labels=garch_labels,
    outdir=Path("..") / "experiments",
    seed=1234,
    sample_size=3500,
    smoothing_grid=[2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250],
    n_series=25,
    burn_in=500,
    alpha=0.1,
    var_warmup=400,
    transform="student",
    base_exp_kwargs={"B": 200, "record_every": 1, "smoothing_method": "ewma"},
    progress=True,
    parallel=True,
    n_jobs=-1,
)

[run_sweeps_for_process_list] Running sweeps for GARCH_base -> saving to ../experiments/GARCH_base


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   31.8s remaining:   47.7s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   32.3s remaining:   21.5s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   32.9s finished
combo runs: 100%|██████████| 1/1 [00:32<00:00, 32.95s/it]


[run_sweeps_for_process_list] Running sweeps for GARCH_trend -> saving to ../experiments/GARCH_trend


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   31.6s remaining:   47.4s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   31.9s remaining:   21.3s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   32.5s finished
combo runs: 100%|██████████| 1/1 [00:32<00:00, 32.55s/it]


[run_sweeps_for_process_list] Running sweeps for GARCH_shock -> saving to ../experiments/GARCH_shock


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   32.5s remaining:   48.8s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   33.0s remaining:   22.0s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   33.6s finished
combo runs: 100%|██████████| 1/1 [00:33<00:00, 33.59s/it]


[run_sweeps_for_process_list] Running sweeps for GARCH_season -> saving to ../experiments/GARCH_season


combo runs:   0%|          | 0/1 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   32.4s remaining:   48.6s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   32.9s remaining:   21.9s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   33.5s finished
combo runs: 100%|██████████| 1/1 [00:33<00:00, 33.52s/it]
